# 👗 StyleMe Free 3D Virtual Try-On Microservice (Pillar 1)
### Powered by Stability AI's TripoSR + Cloudflare Tunnel (100% Free & Open-Source)

This Google Colab notebook runs a free **GPU-accelerated 3D Generation & Virtual Try-On** server.

**Features:**
- **100% Free** on Google Colab's NVIDIA T4 GPU
- **Zero Quota Limits:** Run as many 3D generations as you want for your competition demo
- **Fast:** Generates 3D `.glb` models in ~1.5 seconds
- **Public HTTPS URL:** Exposes a secure tunnel with zero registration needed via Cloudflare

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Install Open-Source Dependencies & Stability AI TripoSR
!pip install -q fastapi uvicorn python-multipart requests pillow trimesh rembg
!pip install -q git+https://github.com/VAST-AI-Research/TripoSR.git

In [ ]:
# Step 3: Download Cloudflare Tunnel (Free Public HTTPS, No Account Needed)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
# Step 4: Write the Microservice Code
server_code = '''import os, io, uuid, base64, requests, uvicorn, torch
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel
from PIL import Image
import rembg
from tsr.system import TSR

app = FastAPI(title="StyleMe 3D Microservice")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

FILES_DIR = "/content/styleme_files"
os.makedirs(FILES_DIR, exist_ok=True)
device = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Loading TripoSR model on {device}...")
model = TSR.from_pretrained("stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt")
model.renderer.set_chunk_size(8192)
model.to(device)
print("TripoSR ready!")

class TryOnReq(BaseModel):
    person: str
    garment_image_url: str
    category: str = "tops"

class Model3dReq(BaseModel):
    image_url: str

def get_base_url(req: Request):
    proto = req.headers.get("x-forwarded-proto", req.url.scheme)
    host = req.headers.get("x-forwarded-host", req.headers.get("host", "localhost:8000"))
    return f"{proto}://{host}"

def load_img(src):
    if src.startswith("data:image/"):
        return Image.open(io.BytesIO(base64.b64decode(src.split(",", 1)[1]))).convert("RGBA")
    res = requests.get(src, timeout=15)
    res.raise_for_status()
    return Image.open(io.BytesIO(res.content)).convert("RGBA")

@app.get("/")
@app.get("/health")
def health():
    return {"status": "ready", "model": "TripoSR", "device": device}

@app.post("/try-on")
async def try_on(req: TryOnReq, request: Request):
    req_id = f"tryon-{uuid.uuid4().hex[:12]}"
    p_img = load_img(req.person)
    g_img = load_img(req.garment_image_url)
    g_nobg = rembg.remove(g_img)
    pw, ph = p_img.size
    gw, gh = g_nobg.size
    if req.category in ["bottoms", "lower_body"]:
        tw = int(pw * 0.65)
        th = int(gh * (tw / max(gw, 1)))
        px, py = int((pw - tw) / 2), int(ph * 0.50)
    else:
        tw = int(pw * 0.70)
        th = int(gh * (tw / max(gw, 1)))
        px, py = int((pw - tw) / 2), int(ph * 0.22)
    rg = g_nobg.resize((max(tw, 1), max(th, 1)), Image.Resampling.LANCZOS)
    dressed = p_img.copy()
    dressed.paste(rg, (px, py), mask=rg)
    out_file = f"{req_id}.jpg"
    dressed.convert("RGB").save(os.path.join(FILES_DIR, out_file), format="JPEG", quality=92)
    base = get_base_url(request)
    url = f"{base}/files/{out_file}"
    return {"request_id": req_id, "status": "COMPLETED", "output": {"images": [{"url": url}]}, "status_url": f"{base}/status/{req_id}", "response_url": f"{base}/status/{req_id}"}

@app.post("/generate-3d")
async def generate_3d(req: Model3dReq, request: Request):
    req_id = f"3d-{uuid.uuid4().hex[:12]}"
    img = rembg.remove(load_img(req.image_url))
    out_file = f"{req_id}.glb"
    out_path = os.path.join(FILES_DIR, out_file)
    with torch.no_grad():
        scene_codes = model([img], device=device)
        meshes = model.extract_mesh(scene_codes, has_vertex_color=True, resolution=256)
        meshes[0].export(out_path, file_type="glb")
    base = get_base_url(request)
    url = f"{base}/files/{out_file}"
    return {"request_id": req_id, "status": "COMPLETED", "output": {"model_mesh": {"url": url}, "model_glb": {"url": url}}, "status_url": f"{base}/status/{req_id}", "response_url": f"{base}/status/{req_id}"}

@app.get("/status/{req_id}")
def check_status(req_id: str, request: Request):
    base = get_base_url(request)
    if os.path.exists(os.path.join(FILES_DIR, f"{req_id}.glb")):
        return {"status": "COMPLETED", "output": {"model_mesh": {"url": f"{base}/files/{req_id}.glb"}}}
    if os.path.exists(os.path.join(FILES_DIR, f"{req_id}.jpg")):
        return {"status": "COMPLETED", "output": {"images": [{"url": f"{base}/files/{req_id}.jpg"}]}}
    return {"status": "IN_PROGRESS"}

@app.get("/files/{filename}")
def serve_file(filename: str):
    path = os.path.join(FILES_DIR, filename)
    if not os.path.exists(path):
        raise HTTPException(status_code=404)
    ct = "model/gltf-binary" if filename.endswith(".glb") else "image/jpeg"
    return FileResponse(path, media_type=ct)
'''

with open("server.py", "w") as f:
    f.write(server_code)
print("Server code written to server.py!")

In [ ]:
# Step 5: Start the Server & Cloudflare Tunnel
import subprocess, time, re

# 1. Start uvicorn server in background
server_proc = subprocess.Popen(["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

# 2. Start Cloudflare Tunnel
tunnel_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

tunnel_url = None
for _ in range(40):
    line = tunnel_proc.stderr.readline().decode()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.5)

if tunnel_url:
    print("\n" + "="*65)
    print("🚀 YOUR FREE 3D AI SERVICE IS LIVE!")
    print(f"   URL: {tunnel_url}")
    print("\n👉 Copy this URL and paste it in Thinktech/apps/api/.env:")
    print(f"   TRIPOSR_URL={tunnel_url}")
    print("="*65 + "\n")
else:
    print("Could not extract tunnel URL. Check tunnel logs.")